In [1]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso,Ridge,ElasticNet,LinearRegression
from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv("car-mpg (1).csv")

In [3]:
df.sample(5)

,mpg,cyl,disp,hp,wt,acc,yr,origin,car_type,car_name
243,21.5,3,80.0,110,2720,13.5,77,3,1,mazda rx-4
139,14.0,8,302.0,140,4638,16.0,74,1,0,ford gran torino (sw)
38,14.0,8,350.0,165,4209,12.0,71,1,0,chevrolet impala
148,26.0,4,116.0,75,2246,14.0,74,2,1,fiat 124 tc
0,18.0,8,307.0,130,3504,12.0,70,1,0,chevrolet chevelle malibu


In [8]:
df = df.drop(columns=['car_type'])

In [9]:
df.head()

,mpg,cyl,disp,hp,wt,acc,yr,origin
0,18.0,8,307.0,130,3504,12.0,70,1
1,15.0,8,350.0,165,3693,11.5,70,1
2,18.0,8,318.0,150,3436,11.0,70,1
3,16.0,8,304.0,150,3433,12.0,70,1
4,17.0,8,302.0,140,3449,10.5,70,1


In [13]:
df.isnull().sum()

mpg       0
cyl       0
disp      0
hp        0
wt        0
acc       0
yr        0
origin    0
dtype: int64

In [29]:
import numpy as np
import pandas as pd

# Replace '?' with NaN and convert the column to numeric
df['hp'] = pd.to_numeric(df['hp'], errors='coerce')

# Calculate the median of hp (ignoring NaNs) and fill missing values
median_hp = df['hp'].median()
df['hp'] = df['hp'].fillna(median_hp)

# Re-define your x and y
x = df.drop(['mpg'], axis=1)
y = df[['mpg']]

In [30]:
# Check which rows contain '?' as a string
for col in x.columns:
    bad_rows = x[x[col] == '?']
    if len(bad_rows) > 0:
        print(f"Found {len(bad_rows)} occurrences in column: {col}")

In [31]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=21)

In [33]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [35]:
lr = LinearRegression()
lr.fit(x_train_scaled,y_train)
y_pred1 = lr.predict(x_test_scaled)
print('Linear Regression r2_score',r2_score(y_test,y_pred1))

Linear Regression r2_score 0.849964447130668


In [58]:
R = Ridge(alpha = 6)
R.fit(x_train_scaled,y_train)
y_pred2 = R.predict(x_test_scaled)
print('Ridge Regression r2_score',r2_score(y_test,y_pred2))

Linear Regression r2_score 0.8511112431115709


In [73]:
L = Lasso(alpha = 0.01)
L.fit(x_train_scaled,y_train)
y_pred2 = L.predict(x_test_scaled)
print('Lasso Regression r2_score',r2_score(y_test,y_pred2))

Lasso Regression r2_score 0.8495913904849567


In [80]:
E = ElasticNet(alpha=0.005,l1_ratio=0.9)
E.fit(x_train_scaled,y_train)
y_pred2 = E.predict(x_test_scaled)
print('ElasticNet Regression r2_score',r2_score(y_test,y_pred2))

ElasticNet Regression r2_score 0.8499048127664842


In [77]:
from sklearn.model_selection import GridSearchCV

In [79]:
elastic_params = {
    'alpha': [0.0001,0.005, 0.001, 0.01, 0.1, 1, 5, 10, 50, 100, 200],
    'l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}
elastic_grid = GridSearchCV(ElasticNet(), elastic_params, scoring='r2', cv=5)
elastic_grid.fit(x_train_scaled, y_train)

print("Best ElasticNet Parameters:", elastic_grid.best_params_)
print("Best ElasticNet R2 Score:", elastic_grid.best_score_)

Best ElasticNet Parameters: {'alpha': 0.01, 'l1_ratio': 0.6}
Best ElasticNet R2 Score: 0.7960871240298001


In [81]:
best_elastic = elastic_grid.best_estimator_
y_pred_elastic = best_elastic.predict(x_test_scaled)
print('Test Set R2 Score:', r2_score(y_test, y_pred_elastic))

Test Set R2 Score: 0.8502559045062004
